In [11]:
import duckdb

con = duckdb.connect('/workspaces/data_engineering_zoomcamp_2026/workhops-ingestion-with-dlt/taxi_pipeline.duckdb', read_only=True)

sql_diagnostics = """
PRAGMA table_info('taxi_data.trips');
"""

sql_payment_values = """
SELECT DISTINCT payment_type FROM taxi_data.trips ORDER BY 1;
"""

sql_main = """
WITH base AS (
  SELECT
    CAST(trip_pickup_date_time AS DATE) AS pickup_date,
    CAST(payment_type AS VARCHAR) AS payment_type_v,
    tip_amt
  FROM taxi_data.trips
)
SELECT
  MIN(pickup_date) AS start_date,
  MAX(pickup_date) AS end_date,
  ROUND(
    100.0 * AVG(
      CASE
        WHEN lower(trim(payment_type_v)) LIKE '%credit%' THEN 1.0
        WHEN trim(payment_type_v) = '1' THEN 1.0
        ELSE 0.0
      END
    ),
    2
  ) AS credit_card_pct,
  ROUND(SUM(tip_amt), 2) AS total_tip_amt
FROM base;
"""

print('=== SQL: diagnostics ===')
print(sql_diagnostics)
print('=== SQL: payment distinct ===')
print(sql_payment_values)
print('=== SQL: main metrics ===')
print(sql_main)

print('\n=== Diagnostics (table_info) ===')
print(con.execute(sql_diagnostics).df())

print('\n=== Distinct payment_type ===')
print(con.execute(sql_payment_values).df())

print('\n=== Metrics ===')
print(con.execute(sql_main).df())

=== SQL: diagnostics ===

PRAGMA table_info('taxi_data.trips');

=== SQL: payment distinct ===

SELECT DISTINCT payment_type FROM taxi_data.trips ORDER BY 1;

=== SQL: main metrics ===

WITH base AS (
  SELECT
    CAST(trip_pickup_date_time AS DATE) AS pickup_date,
    CAST(payment_type AS VARCHAR) AS payment_type_v,
    tip_amt
  FROM taxi_data.trips
)
SELECT
  MIN(pickup_date) AS start_date,
  MAX(pickup_date) AS end_date,
  ROUND(
    100.0 * AVG(
      CASE
        WHEN lower(trim(payment_type_v)) LIKE '%credit%' THEN 1.0
        WHEN trim(payment_type_v) = '1' THEN 1.0
        ELSE 0.0
      END
    ),
    2
  ) AS credit_card_pct,
  ROUND(SUM(tip_amt), 2) AS total_tip_amt
FROM base;


=== Diagnostics (table_info) ===
    cid                    name                      type  notnull dflt_value  \
0     0                 end_lat                    DOUBLE    False       None   
1     1                 end_lon                    DOUBLE    False       None   
2     2                f

In [12]:
sql_latest_load_metrics = """
WITH loads AS (
  SELECT
    load_id,
    inserted_at,
    lower(trim(CAST(status AS VARCHAR))) AS status_v
  FROM taxi_data._dlt_loads
),
latest_loaded AS (
  SELECT load_id, inserted_at, status_v
  FROM loads
  WHERE status_v = 'loaded'
  ORDER BY inserted_at DESC
  LIMIT 1
),
latest_any AS (
  SELECT load_id, inserted_at, status_v
  FROM loads
  ORDER BY inserted_at DESC
  LIMIT 1
),
latest AS (
  SELECT * FROM latest_loaded
  UNION ALL
  SELECT *
  FROM latest_any
  WHERE NOT EXISTS (SELECT 1 FROM latest_loaded)
),
base AS (
  SELECT
    t._dlt_load_id,
    CAST(t.trip_pickup_date_time AS DATE) AS pickup_date,
    CAST(t.trip_dropoff_date_time AS DATE) AS dropoff_date,
    CAST(t.payment_type AS VARCHAR) AS payment_type_v,
    COALESCE(t.tip_amt, 0) AS tip_amt
  FROM taxi_data.trips t
  JOIN latest l
    ON t._dlt_load_id = l.load_id
)
SELECT
  (SELECT load_id FROM latest) AS latest_load_id,
  (SELECT status_v FROM latest) AS latest_status_normalized,
  MIN(pickup_date) AS pickup_start_date,
  MAX(pickup_date) AS pickup_end_date,
  MIN(dropoff_date) AS dropoff_start_date,
  MAX(dropoff_date) AS dropoff_end_date,
  CASE
    WHEN MIN(pickup_date) = MIN(dropoff_date)
     AND MAX(pickup_date) = MAX(dropoff_date)
    THEN 'MATCH'
    ELSE 'DIFFER'
  END AS date_span_check,
  ROUND(
    100.0 * AVG(
      CASE
        WHEN lower(trim(payment_type_v)) LIKE '%credit%' THEN 1.0
        WHEN trim(payment_type_v) = '1' THEN 1.0
        ELSE 0.0
      END
    ),
    2
  ) AS credit_card_pct,
  ROUND(SUM(tip_amt), 2) AS total_tip_amt
FROM base;
"""

sql_cumulative_for_compare = """
SELECT
  ROUND(
    100.0 * AVG(
      CASE
        WHEN lower(trim(CAST(payment_type AS VARCHAR))) LIKE '%credit%' THEN 1.0
        WHEN trim(CAST(payment_type AS VARCHAR)) = '1' THEN 1.0
        ELSE 0.0
      END
    ),
    2
  ) AS cumulative_credit_card_pct,
  ROUND(SUM(COALESCE(tip_amt, 0)), 2) AS cumulative_total_tip_amt
FROM taxi_data.trips;
"""

print('=== Latest-load-only metrics ===')
latest_df = con.execute(sql_latest_load_metrics).df()
print(latest_df)

print('\n=== Cumulative (for difference explanation) ===')
cum_df = con.execute(sql_cumulative_for_compare).df()
print(cum_df)

=== Latest-load-only metrics ===
       latest_load_id latest_status_normalized pickup_start_date  \
0  1771856567.5310662                        0        2009-06-01   

  pickup_end_date dropoff_start_date dropoff_end_date date_span_check  \
0      2009-06-30         2009-06-01       2009-07-01          DIFFER   

   credit_card_pct  total_tip_amt  
0            26.66        6063.41  

=== Cumulative (for difference explanation) ===
   cumulative_credit_card_pct  cumulative_total_tip_amt
0                       26.66                  12126.82


In [13]:
sql_loads_check = """
SELECT
  load_id,
  inserted_at,
  CAST(status AS VARCHAR) AS status_raw
FROM taxi_data._dlt_loads
ORDER BY inserted_at DESC
LIMIT 10;
"""

print('=== Recent _dlt_loads ===')
print(con.execute(sql_loads_check).df())

=== Recent _dlt_loads ===
              load_id                      inserted_at status_raw
0  1771856567.5310662 2026-02-23 14:23:14.129508+00:00          0
1  1771856409.8867915 2026-02-23 14:20:36.594102+00:00          0
